In [105]:
### Cu 003 processing ###


#%% load the packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.ticker import FuncFormatter
from matplotlib import cm
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import defdap.hrdic as hrdic
import defdap.ebsd as ebsd
import defdap.experiment as experiment
from defdap.quat import Quat
from defdap.plotting import MapPlot

from pathlib import Path

import copy 
import pandas as pd
import datetime

from scipy.signal import find_peaks
from scipy.linalg import sqrtm, polar
from scipy import stats
from scipy import interpolate 
from scipy import ndimage
from scipy.optimize import curve_fit


import skimage as ski


import os

# get dictools stuff 
import sys
# sys.path.append("c:/work/hrdic-tools/")
# import dictools

plt.rcParams['svg.fonttype'] = 'none'

%matplotlib qt

In [211]:
def calc_rotations(dic_map):
    # calculate rotations from dic displacement field 

    # extract deformation gradient
    f = dic_map.data.f

    # calculate rotation as ang = (F21 - F12)/2
    rot = (f[0,1,:,:] - f[1,0,:,:])/2

    # centre on mean 
    rot = rot - np.nanmean(rot)

    return rot 


def tangential_displacement_extractor(u,v,start,end,linewidth=5):
    # Extracts the components of a vector field (u,v) along a line defined by start and end
    
    # Returns 
    #   uv_tang_component   : components of vector field tangential to line along the profile
    #   uv_inline_component : components of vector field parallel to line along the profile
    #   profu               : u components in original x-y reference frame
    #   profv               : v components in original x-y reference frame

    # get u and v values along profile
    profu = ski.measure.profile_line(u,np.flip(start),np.flip(end),linewidth=linewidth)
    profv = ski.measure.profile_line(v,np.flip(start),np.flip(end),linewidth=linewidth)

    # resolve stuff onto line using vectors 

    # get line and tangent vectors
    para_vect_dir = np.array(end) - np.array(start)
    para_vect_dir = para_vect_dir/(np.sqrt(para_vect_dir[0]**2 + para_vect_dir[1]**2))

    perp_vect_dir = np.array([-para_vect_dir[1],para_vect_dir[0]])

    # get field vectors along line
    uv_vect = np.array([profu,profv])

    # resolve into components 
    uv_tang_component = np.matmul(perp_vect_dir,uv_vect)
    uv_inline_component = np.matmul(para_vect_dir,uv_vect)

    return uv_tang_component, uv_inline_component, profu, profv

# sigmoid function for fitting to grain boundary steps 
def sigmoid(x, L ,x0, k, b):
    y = L / (1 + np.exp(-k*(x-x0))) + b
    return (y)

def mod_sigmoid(x, L ,x0, k, b):
    y = x*L / (1 + np.exp(-k*(x-x0))) + b
    return (y)

In [3]:
exp = experiment.Experiment()

# load DIC data 
data_dir = Path('./DIC/pyvale/')
dic_frame = experiment.Frame()

# dic_step_list = sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv'))

for dic_file in sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv')):
    hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

# dic_file = dic_step_list[-2]
# hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

hfw = 20.0 # microns
pixelwidth = 2048
pixelsize = hfw/pixelwidth



# calculate rotations    
for inc, dic_map in exp.iter_over_maps('hrdic'):
    # add rotation map to dic

    # calculate rotation
    rot = calc_rotations(dic_map)*180/np.pi

    # add to dic_map
    dic_map.data.add(
        'r_ang', rot,
        unit='°', type='map', order=0,
        plot_params={
            'plot_colour_bar': True,
            'clabel': 'Rotation',
            'cmap': 'RdBu_r'
        }
    )

    


for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.set_scale(pixelsize)
    dic_map.set_crop(left=100,right=100,top=100,bottom=100)
    # dic_map.plot_map('max_shear',vmin=0,vmax=0.01,plot_scale_bar=True)
    print(dic_map)

Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a dat

In [4]:
ebsd_frame = experiment.Frame()
data_dir = Path('.')
ebsd.Map(data_dir / 'Pre_EBSD/map.cpr',
         increment=exp.increments[0], frame=ebsd_frame)

Loaded EBSD data (dimensions: 3727 x 2795 pixels, step size: 0.2 um)


In [5]:
ebsd_map = exp.increments[0].maps['ebsd']
# ebsd_map.set_homog_point()

dic_map = exp.increments[0].maps['hrdic']

# dic_map.set_homog_point(vmin=0,vmax=0.05)

In [6]:
ebsd_frame.homog_points = [(1946, 1565),
 (2443, 1000),
 (1305, 1027),
 (1395, 2225),
 (2572, 2193),
 (1876, 1077),
 (2613, 1497),
 (1822, 2208),
 (1259, 1661),
 (2229, 1260),
 (1641, 1325),
 (1693, 1794),
 (2195, 1762)]

In [7]:
dic_frame.homog_points = [(1582, 1380),
 (2615, 172),
 (238, 226),
 (453, 2748),
 (2882, 2728),
 (1431, 326),
 (2970, 1240),
 (1329, 2732),
 (157, 1568),
 (2170, 731),
 (941, 863),
 (1061, 1854),
 (2100, 1795)]

In [8]:
ebsd_map = exp.increments[0].maps['ebsd']

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.link_ebsd_map(ebsd_map, transform_type="polynomial",order=2)
    # dic_map.link_ebsd_map(ebsd_map, transform_type="affine")

In [9]:
# Figure 2 rotation plot

# plot = MapPlot.create(dic_map,rot,rot,vmin=-5,vmax=5,cmap='RdBu_r',
#                       plot_gbs='pixel',
#                       dilate_boundaries=True,
#                       boundary_colour='black',
#                       plot_colour_bar=True,
#                       clabel = 'In-plane rotation / °',
#                       plot_scale_bar=True                     
#                       )
# plt.savefig('./figures_for_paper/dic_rotation_max_load_with_gbs.png',dpi=1000)


In [10]:
# Figure 2 ipf - x plot 

# ebsd_map.plot_map('orientation',component='IPF_x',plot_gbs='pixel',dilate_boundaries=True,plot_scale_bar=True)#,extents=(2000,6000,1000,2500))
# ax=plt.gca()
# ax.set_xlim([1221,2620])
# ax.set_ylim([2327,930])

# plt.savefig('./figures_for_paper/ipfx_with_gbs.png',dpi=1000)

In [11]:
# Figure 2 max shear plot 

# dic_map.plot_map('max_shear',vmin=0,vmax=0.1,plot_gbs='line',plot_scale_bar=True,dilate_boundaries=True)
# plt.savefig('./figures_for_paper/max_shear_max_load_with_gbs.png',dpi=1000)

In [12]:
# Figure 2 zoomed in plot 
dic_map.plot_map('max_shear',vmin=0,vmax=0.1,plot_gbs=False,plot_scale_bar=True)
ax = plt.gca()
ax.set_xlim([1000,2000])
ax.set_ylim([2000,1000])
plt.savefig('./figures_for_paper/max_shear_max_load_wo_gbs_zoom.png',dpi=1000)

In [13]:
# find special boundaries 
misori_twin = Quat.from_axis_angle([1, 1, 1], 60*np.pi/180)
misori_twin_tol = 1*np.pi/180

# create all symmetric equivalent misorientations
misori_twin_all = []
syms = ebsd_map.primary_phase.crystal_structure.symmetries
for sym_i in syms:
    for sym_j in syms:
        misori_twin_all.append(sym_i.conjugate * misori_twin * sym_j)


# get rid of any duplicates
misori_twin_all = list(set(misori_twin_all))

# calculate neighbour network
ebsd_map.build_neighbour_network()

# loop over all grain boundary segments and check if the misorientation between
# the two grains is within tolerance of the twin misorientation
# store all gbs pairs with misorientation for later us 

all_gb_info = []
twin_gb_info = []
twin_lines = []

for grain1, grain2, b_seg in ebsd_map.neighbour_network.edges.data('boundary'):
    twin = False

    # calculate grain ref orientation
    grain1.calc_average_ori()
    grain2.calc_average_ori()

    misori = grain2.ref_ori * grain1.ref_ori.conjugate

    # # calculate misorientation angle in degrees 
    misori_ang = 2*np.arccos(grain1.ref_ori.mis_ori(grain2.ref_ori,ebsd_map.crystal_sym))*180/np.pi

    # add to list of everything 
    all_gb_info.append([grain1,grain2,misori,misori_ang,b_seg])

    # check if twin and add to twin list
    for misori_twin in misori_twin_all:
        if 2 * np.arccos(misori_twin.dot(misori)) < misori_twin_tol:
            twin = True
            break
    
    if not twin:
        continue
    
    twin_lines.append(b_seg)
    twin_gb_info.append([grain1,grain2,misori,misori_ang,b_seg])




# make into boundary set
s_bounds = ebsd.BoundarySet.from_boundary_segments(twin_lines)

Finished building quaternion array (0:00:21) 
Finished finding grain boundaries (0:01:08) 
Finished finding grains (0:01:28) 
Finished constructing neighbour network (0:01:36) 


In [14]:
# data cubifier 
# make the individual maps into a data cube nx * ny * n_DIC_steps 

def datacubifier(exp):
    # find number of steps 
    n_steps = len(exp.increments)

    # get data shape 
    ny,nx = exp.increments[0].maps['hrdic'].shape

    # make empty arrays to fill things with 
    ems_cube    = np.zeros((ny,nx,n_steps))
    e11_cube    = np.zeros((ny,nx,n_steps))
    e22_cube    = np.zeros((ny,nx,n_steps))
    e12_cube    = np.zeros((ny,nx,n_steps))
    r_ang_cube  = np.zeros((ny,nx,n_steps))
    u_cube      = np.zeros((ny,nx,n_steps))
    v_cube      = np.zeros((ny,nx,n_steps))
    f11_cube    = np.zeros((ny,nx,n_steps))
    f12_cube    = np.zeros((ny,nx,n_steps))
    f21_cube    = np.zeros((ny,nx,n_steps))
    f22_cube    = np.zeros((ny,nx,n_steps))

    for inc, dic_map in exp.iter_over_maps('hrdic'):
        ems_cube[:,:,inc]   = dic_map.data['max_shear']
        e11_cube[:,:,inc]   = dic_map.data['e'][0,0]
        e22_cube[:,:,inc]   = dic_map.data['e'][1,1]
        e12_cube[:,:,inc]   = dic_map.data['e'][0,1]
        r_ang_cube[:,:,inc] = dic_map.data['r_ang']
        u_cube[:,:,inc]     = dic_map.data['displacement'][0]
        v_cube[:,:,inc]     = dic_map.data['displacement'][1]
        f11_cube[:,:,inc]   = dic_map.data['f'][0,0]
        f12_cube[:,:,inc]   = dic_map.data['f'][0,1]
        f21_cube[:,:,inc]   = dic_map.data['f'][1,0]
        f22_cube[:,:,inc]   = dic_map.data['f'][1,1]
        



    return ems_cube, e11_cube, e22_cube, e12_cube, r_ang_cube, u_cube, v_cube, f11_cube, f12_cube, f21_cube, f22_cube

# cubify data for more simple processing
ems_cube, e11_cube, e22_cube, e12_cube, r_ang_cube, u_cube, v_cube, f11_cube, f12_cube, f21_cube, f22_cube = datacubifier(exp)
e11_glob = np.nanmean(e11_cube,axis=(0,1))

In [15]:
# plot misorientation distribution function 

# sanity check for all twin boundaries having a MO ~ 60 degrees 
twin_mo_angle = []
for i in range(len(twin_gb_info)):
    twin_mo_angle.append(twin_gb_info[i][3])



all_mo_angle = []
for i in range(len(all_gb_info)):
    all_mo_angle.append(all_gb_info[i][3])

fig,ax = plt.subplots()
ax.hist(all_mo_angle,range=(0,65),bins=50,density=True,color='yellowgreen',edgecolor='olivedrab')
ax.set_ylabel('Probability density / -')
ax.set_xlabel('Misorientation angle / °')
ax.set_aspect(aspect=100)
ax.grid()
plt.tight_layout()

In [ ]:
plt.figure()
plt.plot(twin_mo_angle,'o')

Rotation "intensity" i.e. rotation gradient magnitude

In [ ]:
# this is the spatial variation in in-plane rotation. This shows that the nature of rotations around grain boundaries is profoundly different to slip

# calculate rotations gradients on rotation cubes 
rot_grad_cube = np.sqrt(np.square(np.gradient(r_ang_cube,dic_map.binning,axis=0)) + np.square(np.gradient(r_ang_cube,dic_map.binning,axis=1)))


In [ ]:
plt.figure()
plt.imshow(rot_grad_cube[:,:,4],vmin=0,vmax=0.1)

In [ ]:
fig,ax = plt.subplots()
ax.hist(rot_grad_cube[:,:,0].flatten(),bins=100,range=(0,10))
# ax.set_yscale('log')

In [ ]:

# set consistent colours
gb_colour   = 'darkturquoise'
sb_colour   = 'tomato'
ogb_colour  = 'darkorange'
gc_colour   = 'seagreen'

# percentile line
pc = 99.99

logbins = np.logspace(-3,1,100)

fig,ax=plt.subplots(3,1,sharex=True,sharey=True)
fig.set_size_inches(5,8)

# axis 0
ax[0].set_xlim(1e-3,1)
ax[0].set_ylim(1e-5,1000)

sub_fig_labels = ['(a)','(b)','(c)']

for i,idx in enumerate([1,2,3]):

    tdata = copy.deepcopy(rot_grad_cube[:,:,step_idxs[idx]])

    ems_gbs = tdata[gbs]#/np.nanmean(tdata[gbs])
    ems_sbs = tdata[sbs]#/np.nanmean(tdata[sbs])
    ems_ogbs = tdata[ogbs]#/np.nanmean(tdata[ogbs])
    ems_gcs = tdata[np.invert(gbs)]#/np.nanmean(tdata[np.invert(gbs)])
    

    ax[i].hist(ems_gbs,logbins,density=True,histtype='step',color=gb_colour)
    ax[i].hist(ems_sbs,logbins,density=True,histtype='step',color=sb_colour)
    ax[i].hist(ems_ogbs,logbins,density=True,histtype='step',color=ogb_colour)
    ax[i].hist(ems_gcs,logbins,density=True,histtype='step',color=gc_colour)

    # # add some vertical lines
    ax[i].axvline(np.nanpercentile(ems_gbs,pc),linestyle='--',color=gb_colour)
    ax[i].axvline(np.nanpercentile(ems_sbs,pc),linestyle='--',color=sb_colour)
    ax[i].axvline(np.nanpercentile(ems_ogbs,pc),linestyle='--',color=ogb_colour)
    ax[i].axvline(np.nanpercentile(ems_gcs,pc),linestyle='--',color=gc_colour)

    # annotate 
    glob_e11 = e11_glob[step_idxs[idx]]*100

    
    ax[i].annotate(sub_fig_labels[i] + ' $ϵ_{xx}$ = '+str(glob_e11.round(2))+'%',
                   xy=(ax[0].get_xlim()[0]*1.2,ax[0].get_ylim()[0]*2),
                   fontsize=14,
                   bbox=dict(boxstyle="square,pad=0.1",
                             fc='white',
                             ec='white'))

    ax[i].set_xscale('log')
    ax[i].set_yscale('log')


    ax[i].grid()
    ax[i].grid(which='minor',color='0.9')

    


ax[-1].set_xlabel('Effective strain / -')
ax[1].set_ylabel('Probability density / -')



ax[0].legend(['All GBs','Σ3GBs', 'Other GBs','Grain cores'])
plt.tight_layout()

Small detour into schmid factors

In [ ]:
# plot grain average schmid factors first 
load_vector  = [1,0,0]

# calculate schmid factor pairs 
all_gb_sfs= []
for i in range(len(all_gb_info)):
    # get grains
    g0 = all_gb_info[i][0]
    g1 = all_gb_info[i][1]

    # calc schmid factors
    for g in [g0,g1]:
        g.calc_average_schmid_factors(load_vector)
    
    # max schmid factors
    sfmax0 = np.nanmax(g0.average_schmid_factors)
    sfmax1 = np.nanmax(g1.average_schmid_factors)
    
    all_gb_sfs.append([sfmax0,sfmax1])


# plot grain average schmid factors 
ebsd_map.plot_average_grain_schmid_factors_map(directions=load_vector,vmin=0,vmax=0.5)

# overlay with differences in schmid factor 
ax = plt.gca()

for idx in range(len(all_gb_info)):
    # print(idx)
    bseg = all_gb_info[idx][-1]
    
    # check if boundary bits are empty 
    try:
        bseg_x = np.hstack([np.array(bseg.boundary_points_x)[:,0],np.array(bseg.boundary_points_y)[:,0]])
        bseg_y = np.hstack([np.array(bseg.boundary_points_x)[:,1],np.array(bseg.boundary_points_y)[:,1]])

        # max schmid factors
        this_sfs = abs(np.diff(all_gb_sfs[idx]))

        boundary_colours = ax.scatter(bseg_x,bseg_y,c=this_sfs*np.ones(len(bseg_x)),cmap='inferno',vmin=0,vmax=0.3,s=0.1)
    except:
        print('something weird happened')


bar2 = plt.colorbar(boundary_colours)

ax.set_xlim([1660,2140])
ax.set_ylim([1860,1390])

In [ ]:
# plot histogram of schmid factors and difference in schmid factors for pairs of grains 
grain_avg_sfs = []

for i in range(len(ebsd_map.grains)):
    grain_avg_sfs.append(np.nanmax(ebsd_map[i].average_schmid_factors))


fig, ax = plt.subplots(2,1,sharex=True,height_ratios=[3,5])
ax[1].hist(grain_avg_sfs,bins=25,color='lightgrey',edgecolor='darkgrey')

ymin = 0
ymax = ax[1].get_ylim()[-1]
# ax[1].vlines(np.nanpercentile(grain_avg_sfs,q=[25,50,75]),ax[1].get_ylim()[0],ax[1].get_ylim()[-1],'purple')

ax[1].set_ylim(ymin,ymax)
ax[1].set_xlabel('Grain maximum Schmid factor / -')
ax[1].set_ylabel('# Grains')
ax[1].grid('on')

ax[0].boxplot(grain_avg_sfs,orientation='horizontal',flierprops=dict(alpha=0.2,markersize=0.2))
ax[0].axis('off')

plt.tight_layout()


# Schmid factor difference
fig, ax = plt.subplots(2,1,sharex=True,height_ratios=[3,5])
ax[1].hist(abs(np.diff(all_gb_sfs)),bins=25,color='lightgrey',edgecolor='darkgrey')

ymin = 0
ymax = ax[1].get_ylim()[-1]
# ax.vlines(np.nanpercentile(abs(np.diff(all_gb_sfs)),q=[25,50,75]),ax.get_ylim()[0],ax.get_ylim()[-1],'purple')

ax[1].set_ylim(ymin,ymax)
ax[1].set_xlabel('Neighbouring grain Schmid factor difference / -')
ax[1].set_ylabel('# Grain boundaries')
ax[1].grid('on')

ax[0].boxplot(abs(np.diff(all_gb_sfs)),orientation='horizontal',flierprops=dict(alpha=0.2,markersize=0.2))
ax[0].axis('off')

plt.tight_layout()


In [ ]:
# loop over all grains to get schmid factors
all_gb_sfs= []
load_vector  = [1,0,0]

for i in range(len(all_gb_info)):
    # get grains
    g0 = all_gb_info[i][0]
    g1 = all_gb_info[i][1]

    # calc schmid factors
    for g in [g0,g1]:
        g.calc_average_schmid_factors(load_vector)
    
    # max schmid factors
    sfmax0 = np.nanmax(g0.average_schmid_factors)
    sfmax1 = np.nanmax(g1.average_schmid_factors)
    
    all_gb_sfs.append([sfmax0,sfmax1])


plt.figure()
for idx in range(len(all_gb_info)):
    # print(idx)
    bseg = all_gb_info[idx][-1]
    
    # check if boundary bits are empty 
    try:
        bseg_x = np.hstack([np.array(bseg.boundary_points_x)[:,0],np.array(bseg.boundary_points_y)[:,0]])
        bseg_y = np.hstack([np.array(bseg.boundary_points_x)[:,1],np.array(bseg.boundary_points_y)[:,1]])

        plt.scatter(bseg_x,bseg_y,c=all_gb_info[idx][3]*np.ones(len(bseg_x)),cmap='turbo',vmin=0,vmax=65,s=0.1)
    except:
        print('something weird happened')

ax = plt.gca()
plt.colorbar()
ax.set_aspect('equal')

ax.set_xlim([1660,2140])
ax.set_ylim([1860,1390])

plt.figure()
for idx in range(len(all_gb_info)):
    # print(idx)
    bseg = all_gb_info[idx][-1]
    
    # check if boundary bits are empty 
    try:
        bseg_x = np.hstack([np.array(bseg.boundary_points_x)[:,0],np.array(bseg.boundary_points_y)[:,0]])
        bseg_y = np.hstack([np.array(bseg.boundary_points_x)[:,1],np.array(bseg.boundary_points_y)[:,1]])

        # max schmid factors
        this_sfs = abs(np.diff(all_gb_sfs[idx]))

        plt.scatter(bseg_x,bseg_y,c=this_sfs*np.ones(len(bseg_x)),cmap='turbo',vmin=0,vmax=0.3,s=0.1)
    except:
        print('something weird happened')

ax = plt.gca()
# plt.colorbar()
ax.set_aspect('equal')

ax = plt.gca()
# ax.set_xlim([500,3000])
# ax.set_ylim([2500,500])

ax.set_xlim([1660,2140])
ax.set_ylim([1860,1390])

In [ ]:
bseg = all_gb_info[0][-1]
    
try:
    bseg_x = np.hstack([np.array(bseg.boundary_points_x)[:,0],np.array(bseg.boundary_points_y)[:,0]])
    bseg_y = np.hstack([np.array(bseg.boundary_points_x)[:,1],np.array(bseg.boundary_points_y)[:,1]])
except:
    pass

bseg = all_gb_info[0]

The good stuff - actually measuring GB sliding

In [46]:


min_gb_size = 20

gb_number_image = np.empty(ebsd_map.shape)
gb_number_image[:] = 0

gb_misori_image = np.empty(ebsd_map.shape)
gb_misori_image[:] = 0

for idx in range(len(all_gb_info)):

    bseg = all_gb_info[idx][-1]
    
    try:
        bseg_x = np.hstack([np.array(bseg.boundary_points_x)[:,0],np.array(bseg.boundary_points_y)[:,0]])
        bseg_y = np.hstack([np.array(bseg.boundary_points_x)[:,1],np.array(bseg.boundary_points_y)[:,1]])

        if len(bseg_x) >= min_gb_size: 

            gb_number_image[bseg_y,bseg_x] = idx
            gb_misori_image[bseg_y,bseg_x] = all_gb_info[idx][3]
    except:
        pass


gb_number_image = dic_map.warp_to_dic_frame(gb_number_image,order=0)

gb_misori_image = dic_map.warp_to_dic_frame(gb_misori_image,order=0)

plt.figure()
plt.imshow(gb_number_image)

plt.figure()
plt.imshow(gb_misori_image)




c:\Users\bepoole\AppData\Local\miniforge3\envs\gb_sliding_paper\Lib\site-packages\defdap\experiment.py:67: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PolynomialTransform.from_estimate` class constructor instead.
  transform.estimate(


In [47]:
#dilation footprint size
fp_size = 7
fp = ski.morphology.disk(fp_size)

gb_number_image = ski.morphology.dilation(gb_number_image,fp) # something wrong here with shifting
gb_misori_image = ski.morphology.dilation(gb_misori_image,fp) # something wrong here with shifting

gb_number_image[gb_number_image < 1] = np.nan
gb_misori_image[gb_misori_image < 0.001] = np.nan

plt.figure()
plt.imshow(gb_misori_image,vmin=0,vmax=60,cmap='turbo')

# plt.gca().format_cursor_data()

In [48]:
gb_idx = 7821

xlim = [1550,1700]
ylim = [1550,1350]

plt.figure()
plt.imshow(gb_number_image,vmin=1,cmap='turbo')
plt.xlim(xlim)
plt.ylim(ylim)

testim = copy.deepcopy(r_ang_cube[:,:,4])

plt.figure()
plt.imshow(testim,vmin=-0.5,vmax=0.5,cmap='RdBu_r')
plt.xlim(xlim)
plt.ylim(ylim)

testim[gb_number_image != gb_idx] = np.nan

this_mo = np.nanmean(gb_misori_image[gb_number_image == gb_idx])

plt.figure()
plt.imshow(testim,vmin=-0.5,vmax=0.5,cmap='RdBu_r')
plt.xlim(xlim)
plt.ylim(ylim)



(1550.0, 1350.0)

In [315]:
# tangent point spacing
tangent_point_spacing = 2 #um 
tangent_line_length = 5 #um
gbs_fit_profile_gradient_factor = 5 # i.e. the gradient fitted to the discontinuity is at least 10 times larger that that on the other sides 

step_idx = 20

fig_map,ax_map = plt.subplots()

ax_map.imshow(ems_cube[:,:,step_idx],vmin=0,vmax=0.1,cmap='viridis',alpha=1)

[yidx,xidx] = np.mgrid[0:dic_map.shape[0],0:dic_map.shape[1]]

gbxidx = xidx[gb_number_image == gb_idx]


gbyidx = yidx[gb_number_image == gb_idx]

# plt.plot(gbxidx,gbyidx,'+')

spl,p = interpolate.make_splprep([gbyidx,gbxidx],s=1000000)
p_rough = np.linspace(0,1,10)

# calculate rough profile so we can get length of GB segment 
fpointsy_rough,fpointsx_rough = spl(p_rough)

fpoints_rough = [fpointsx_rough,fpointsy_rough]

gb_seg_length = [np.sqrt(sum((q - p) ** 2 for p, q in zip(p1, p2))) for p1, p2 in zip(fpoints_rough, fpoints_rough[1:])][0]
gb_seg_length = gb_seg_length*dic_map.scale


# get the actual sampling positions we want 
p_fit = np.linspace(0,1,int(gb_seg_length//tangent_point_spacing))

fpointsy,fpointsx = spl(p_fit)

# ax_map.plot(fpointsx,fpointsy,'yo')

# calculate gradient
# derivative spline
spl_der = spl.derivative(nu=1)

# dx/dp and dy/dp
dydp,dxdp = spl_der(p_fit)

# dy/dx 
grad = dydp/dxdp
dx = np.ones(fpointsx.shape)
dy = dx*grad

# tangent vector of the line 
ltang = np.vstack((dx,dy))

# normalise
ltang = ltang / np.linalg.norm(ltang,axis=0)

# normal to line 
lnorm = np.vstack((dx,-1*dx/grad))

# normalise 
lnorm = lnorm / np.linalg.norm(lnorm,axis=0)

fig_disp,ax_disp = plt.subplots()

fig_gbs,ax_gbs = plt.subplots()

# now we need to loop over each point and extract a line profile
for tang_point_idx in range(0,len(p_fit)):
    # get point on the line 
    x0 = fpointsx[tang_point_idx]
    y0 = fpointsy[tang_point_idx]

    [x_end,y_end] = [x0,y0] + 0.5*(tangent_line_length/dic_map.scale)*lnorm[:,tang_point_idx]
    [x_start,y_start] = [x0,y0] - 0.5*(tangent_line_length/dic_map.scale)*lnorm[:,tang_point_idx]

    ax_map.plot([x_start,x_end],[y_start,y_end])


    uv_tang,_,_,_ = tangential_displacement_extractor(u_cube[:,:,step_idx],v_cube[:,:,step_idx],[x_start,y_start],[x_end,y_end])

    ax_disp.plot(uv_tang,'-')


    # now we need to extract the step at the gb from this curve, fit a three piece linear fit 
    x_line = np.linspace(0,len(uv_tang),len(uv_tang))

    #pop out nans 
    x_line = x_line[~np.isnan(uv_tang)]
    uv_tang = uv_tang[~np.isnan(uv_tang)]
    

    p0 = [np.mean(x_line) - 2, np.mean(x_line) + 2, 0.1, 10, 0.1, uv_tang[0]] # this is an mandatory initial guess

    try:
        popt, pcov = curve_fit(pw_linear, x_line, uv_tang,p0, method='lm')

        uv_fit = pw_linear(x_line,*popt)

        ax_disp.plot(uv_fit,'+')

        # check for gb sliding in profile
        if (abs(popt[3]) > 10*abs(popt[2])) & (abs(popt[3]) > 10*abs(popt[4])) or np.sign(popt[3]) != np.sign(popt[2]) or np.sign(popt[3]) != np.sign(popt[4]):
            print('GB sliding detected')

            gb_sliding_distance = abs(pw_linear(popt[0],*popt) - pw_linear(popt[1],*popt))

    except:

        gb_sliding_distance = np.nan


    # scale into real units 
    gb_sliding_distance = gb_sliding_distance*dic_map.scale

    ax_gbs.plot(tang_point_idx,gb_sliding_distance,'+')

    ax_map.scatter(x0,y0,c=gb_sliding_distance,cmap='inferno',vmin=0,vmax=0.2,s=100)


GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected
GB sliding detected


In [297]:
m0 = popt[2]
m1 = popt[3]
m2 = popt[4]

factor = 10

(abs(m1) > 10*abs(m0)) & (abs(m1) > 10*abs(m2))

np.True_

In [300]:
plt.figure()
plt.plot(x_line,uv_tang)
plt.plot(x_line,pw_linear(x_line,*popt))

gb_sliding = abs(pw_linear(popt[0],*popt) - pw_linear(popt[1],*popt))
gb_sliding

np.float64(0.4007966533190891)

In [195]:
from scipy.signal import detrend

In [244]:
plt.figure()
plt.plot(uv_tang)



p0 = [1, 2, 0, 1, 0, 1] # this is an mandatory initial guess

popt, pcov = curve_fit(pw_linear, x_line, uv_tang,p0, method='lm')

y_linear = pw_linear(x_line,*popt)

plt.plot(y_linear,'+')

In [215]:
plt.figure()
plt.plot(x_line,x_line*-1*sigmoid(x_line,1,25,1,1) + x_line*2*sigmoid(x_line,1,25,2,1) )

In [ ]:
# make up our own function! 

# three linear bits 
x01 = 1
x12 = 2
m0 = 0
m1 = 1 
m2 = 0
c = 1

p = [x01, x12, m0, m1, m2, c]
p0 = [1, 2, 0, 1, 0, 1]
def pw_linear(x, x01, x12, m0, m1, m2, c):


    y = (
        (x < x01)                   * (m0*(x - x01) + c) + 
        ((x >= x01) & (x < x12))    * (m1*(x - x01) + c) + 
        (x >= x12)                  * (m2*(x - x12) + m1*(x12 - x01) + c)
    )

    return y

x = np.linspace(0,5,100)

plt.figure()
plt.plot(x,pw_linear(x,*p))

In [ ]:
ems = []
mo = []

ems_slice = copy.copy(abs(r_ang_cube[:,:,10]))


for gb_idx in range(len(all_gb_info)):
    
    # get ems from gb
    ems.append(np.nanpercentile(ems_slice[gb_number_image == gb_idx],75))
    mo.append(all_gb_info[gb_idx][3])

Figure 3 - grain boundary sliding in early deformation

In [ ]:
# Plot the ipf-x map for this zoomed in region


ebsd_map.plot_ipf_map([1,0,0],plot_gbs='line',dilate_boundaries=True,plot_scale_bar=True)
ax = plt.gca()
ax.set_xlim([1660,2140])
ax.set_ylim([1860,1390])

# plt.savefig('figures_for_paper/Figure_3/ipf_x_zoom.png',dpi=1000)

In [ ]:
# let's find some areas where we've got some gb sliding early on 

# indices of steps we want to use
step_idxs = [2,3,4,7,10]

# number of steps we want to consider 
num_steps = len(step_idxs)

# range on maps
xrange = [1000,2000]
yrange = [2000,1000]
ems_crange = [0,0.02]
r_ang_crange = [-1,1]

# make figure
fig = plt.figure()
fig.set_size_inches([len(step_idxs)*2,4])
gs = gridspec.GridSpec(2,num_steps+1,width_ratios=np.append(np.ones((1,num_steps)),0.05))


for col_idx,step_idx in enumerate(step_idxs):

    ax0 = fig.add_subplot(gs[0,col_idx])
    ems_colours = ax0.imshow(ems_cube[:,:,step_idx],
            vmin=ems_crange[0],
            vmax=ems_crange[1])

    ax1 = fig.add_subplot(gs[1,col_idx])
    r_ang_colours = ax1.imshow(r_ang_cube[:,:,step_idx],
                vmin=r_ang_crange[0],
                vmax=r_ang_crange[1],
                cmap='RdBu_r')
    
    for ax in [ax0,ax1]:
        ax.set_xlim(xrange)
        ax.set_ylim(yrange)
        ax.set_xticks([])
        ax.set_yticks([])

    # get global x strain for title 
    glob_e11 = e11_glob[step_idx]*100

    ax0.set_title('$ϵ_{xx}$ = '+str(glob_e11.round(2))+'%')

# colourbars 
ax0 = fig.add_subplot(gs[0,-1])
fig.colorbar(ems_colours,cax=ax0,ticks=[ems_crange[0],np.mean(ems_crange),ems_crange[1]],label='Effective strain / -')

ax1 = fig.add_subplot(gs[1,-1])
fig.colorbar(r_ang_colours,cax=ax1,ticks=[r_ang_crange[0],0,r_ang_crange[1]],label='In-plane rotation / °')


plt.tight_layout()
# plt.savefig('figures_for_paper/Figure_3/gb_sliding_progression.png',dpi=1000)

In [ ]:
# mask thing by gb vs non-gb etc 

ebsd_map.data['grain_boundaries']

dilation_factor = 2
cleaning_footprint = 2
# dilate normal boundaries 
gbs = ski.morphology.binary_dilation(dic_map.data['grain_boundaries'].image,np.ones((cleaning_footprint,cleaning_footprint)))
sbs = ski.morphology.binary_dilation(dic_map.warp_to_dic_frame(s_bounds.image),np.ones((cleaning_footprint,cleaning_footprint)))

# xor to get other boundaries
ogbs = gbs ^ sbs

# boundary opening to destroy any tiny bits left 
ogbs = ski.morphology.binary_opening(ogbs,np.ones((cleaning_footprint,cleaning_footprint)))

# final dilation 
gbs     = ski.morphology.binary_dilation(gbs,np.ones((dilation_factor,dilation_factor)))
sbs     = ski.morphology.binary_dilation(sbs,np.ones((dilation_factor,dilation_factor)))
ogbs    = ski.morphology.binary_dilation(ogbs,np.ones((dilation_factor,dilation_factor)))

ogbs = ski.morphology.binary_opening(ogbs,np.ones((dilation_factor,dilation_factor)))




In [ ]:

# plot images to show this process
fig,ax = plt.subplots(1,3)
fig.set_size_inches(8,3)
ax[0].imshow(gbs,cmap='bone_r')
ax[0].set_title('All grain boundaries')


ax[1].imshow(sbs,cmap='bone_r')
ax[1].set_title('Σ3 grain boundaries')

ax[2].imshow(ogbs,cmap='bone_r')
ax[2].set_title('Other grain boundaries')


for axs in ax:
    axs.set_xticks([])
    axs.set_yticks([])

plt.tight_layout()

# plt.savefig('figures_for_paper/Figure_4/GB_decomposition.png',dpi=1000)

In [ ]:

# plt.figure()
# plt.imshow(test_box)


idx = 4
plt.figure()
# plt.imshow(r_ang_cube[:,:,idx],vmin=-0.5,vmax=0.5,cmap='RdBu_r')
plt.imshow(ems_cube[:,:,idx],vmin=0,vmax=0.1)
# plt.imshow(test_boxu)

#select roi

roixy = plt.ginput(2,show_clicks=True)
plt.plot(roixy[0][0],roixy[0][1],'rx')
plt.plot(roixy[1][0],roixy[1][1],'rx')


xlim = np.sort([int(roixy[0][0]),int(roixy[1][0])])
ylim = np.sort([int(roixy[0][1]),int(roixy[1][1])])



# select lines 
plt.figure()
# plt.imshow(r_ang_cube[:,:,idx],vmin=-0.5,vmax=0.5,cmap='RdBu_r',alpha=0.5)
plt.imshow(ems_cube[:,:,idx],vmin=0,vmax=0.1)
plt.xlim([xlim[0],xlim[1]])
plt.ylim([ylim[1],ylim[0]])

# click pairs of points - one on GB, one in grain. The code will then mirror this to the other side! Click GB first 
n_pairs = 5

# line half lengths - in microns 
prof_half_length = 3
cmap = mpl.colormaps['Dark2'].colors
line_colours = cmap[0:n_pairs]

prof_half_length /= (dic_map.scale)


linexy = plt.ginput(2*n_pairs,show_clicks=True)
linexy = np.array(linexy)

# store line positions 
line_gb = linexy[0::2,:]
line_ends = linexy[1::2,:]

# get line gradients 


line_starts = np.zeros(line_ends.shape)

for i in range(0,n_pairs):
    

    m = (line_ends[i,1] - line_gb[i,1])/(line_ends[i,0] - line_gb[i,0])

    line_starts[i,0]    = line_gb[i,0] - prof_half_length/np.sqrt(1 + m**2)
    line_ends[i,0]      = line_gb[i,0] + prof_half_length/np.sqrt(1 + m**2)

    line_starts[i,1]    = line_gb[i,1] + m*(line_starts[i,0]-line_gb[i,0])
    line_ends[i,1]    = line_gb[i,1] + m*(line_ends[i,0]-line_gb[i,0])

    plt.plot([line_starts[i,0],line_ends[i,0]],[line_starts[i,1],line_ends[i,1]],'x-',color = line_colours[i])
    plt.plot([line_gb[i,0],line_gb[i,1],'*'])

u_test = copy.deepcopy(u_cube[:,:,idx])
v_test = copy.deepcopy(v_cube[:,:,idx])
ems_test = copy.deepcopy(ems_cube[:,:,idx])

In [ ]:
plt.figure()
# plt.imshow(r_ang_cube[:,:,idx],vmin=-0.5,vmax=0.5,cmap='RdBu_r',alpha=0.5)
plt.imshow(ems_cube[:,:,idx],vmin=0,vmax=0.1)
plt.xlim([xlim[0],xlim[1]])
plt.ylim([ylim[1],ylim[0]])
line_starts = np.zeros(line_ends.shape)

for i in range(0,n_pairs):
    

    m = (line_ends[i,1] - line_gb[i,1])/(line_ends[i,0] - line_gb[i,0])

    line_starts[i,0]    = line_gb[i,0] - prof_half_length/np.sqrt(1 + m**2)
    line_ends[i,0]      = line_gb[i,0] + prof_half_length/np.sqrt(1 + m**2)

    line_starts[i,1]    = line_gb[i,1] + m*(line_starts[i,0]-line_gb[i,0])
    line_ends[i,1]    = line_gb[i,1] + m*(line_ends[i,0]-line_gb[i,0])

    plt.plot([line_starts[i,0],line_ends[i,0]],[line_starts[i,1],line_ends[i,1]],'x-',color = line_colours[i])
    plt.plot([line_gb[i,0],line_gb[i,1],'*'])

In [ ]:
# now we need to resolve them into the tangential displacements along the line 
# get line gradient 
step_idxs = [2,3,4,7,10]

ylimits = [-1,1]

offsets = np.linspace(-0.5,0.5,len(step_idxs))

for count,idx in enumerate(step_idxs):

    u_test = copy.deepcopy(u_cube[:,:,idx])
    v_test = copy.deepcopy(v_cube[:,:,idx])
    ems_test = copy.deepcopy(ems_cube[:,:,idx])

    fig,ax = plt.subplots()
    for i in range(0,n_pairs):

        uv_tang_component, uv_inline_component, profu, profv = tangential_displacement_extractor(u_test,v_test,line_starts[i,:],line_ends[i,:])

        # scale to real units 
        uv_tang_component *= dic_map.scale#/dic_map.binning
        uv_inline_component *= dic_map.scale#/dic_map.binning

        uv_tang_component -= np.mean(uv_tang_component)
        uv_inline_component -= np.mean(uv_inline_component)

        # get physical size of line with corresponding profile coordinates 
        pos_prof = np.linspace(0,len(uv_tang_component),len(uv_tang_component)) * dic_map.scale

        # offset so it's easier to read 
        uv_tang_component += offsets[i]
    
        ax.plot(pos_prof,uv_tang_component,'-',color=line_colours[i])
        ax.set_ylabel('Tangential displacement / μm')
        
        ax.set_ylim(ylimits)
        ax.set_xlabel('Profile position / μm')

        plt.title('E11 = ' + str(np.round(e11_glob[count]*100,2)) + '%')

    
    
    plt.tight_layout()
    # plt.savefig('boundary_slide' + str(int(idx)) +'.png')

In [ ]:
# vector plot, not 100% sure this works well

idx = -1
plt.figure()
plt.imshow(ems_cube[:,:,idx],vmin=0,vmax=0.1)

xy = plt.ginput(2,show_clicks=True)
plt.plot(xy[0][0],xy[0][1],'rx')
plt.plot(xy[1][0],xy[1][1],'rx')

xlim = np.sort([int(xy[0][0]),int(xy[1][0])])
ylim = np.sort([int(xy[0][1]),int(xy[1][1])])



u_test = copy.deepcopy(u_cube[ylim[0]:ylim[-1],xlim[0]:xlim[-1],idx])
v_test = copy.deepcopy(v_cube[ylim[0]:ylim[-1],xlim[0]:xlim[-1],idx])
ems_test = copy.deepcopy(ems_cube[ylim[0]:ylim[-1],xlim[0]:xlim[-1],idx])

# # normalise each set of vectors 
# u_test -= np.nanmean(u_test)
# v_test -= np.nanmean(v_test)

# reduce so it's not as crazy for the vectors 
# set this so that there are a roughly fixed number per image width
arrows_per_image = 20

reduction_factor = int(abs(xlim[-1] - xlim[0])/arrows_per_image)

# reduction_factor = 5
u_vect = u_test[::reduction_factor,::reduction_factor]
v_vect = v_test[::reduction_factor,::reduction_factor]

# centralise so that they're displacements relative to the centre of the map
u_vect_centre = u_vect[u_vect.shape[0]//2,u_vect.shape[1]//2]
v_vect_centre = v_vect[v_vect.shape[0]//2,v_vect.shape[1]//2]

u_vect = u_vect - u_vect_centre
v_vect = v_vect - v_vect_centre
uvmag = np.sqrt(u_vect**2 + v_vect**2)

# normalise mag
u_vect = u_vect/uvmag
v_vect = v_vect/uvmag

x = np.linspace(0,u_test.shape[1],u_vect.shape[1])
y = np.linspace(0,u_test.shape[0],u_vect.shape[0])
# y = np.flipud(y)

plt.figure()
plt.imshow(ems_test,vmin=0,vmax=0.1,alpha=0.5)
plt.quiver(x,y,u_vect,v_vect,pivot='tail',scale=0.5,angles='xy',scale_units = 'xy')

In [ ]:
# what about the level of deformation at grain boundaries 
idx = 10

xlim = [1500,1700]
ylim = [1350,1450]

# xlim = [1500,1550]
# ylim = [1300,1400]



u_test = copy.deepcopy(u_cube[ylim[0]:ylim[-1],xlim[0]:xlim[-1],idx])
v_test = copy.deepcopy(v_cube[ylim[0]:ylim[-1],xlim[0]:xlim[-1],idx])
ems_test = copy.deepcopy(ems_cube[ylim[0]:ylim[-1],xlim[0]:xlim[-1],idx])

# # normalise each set of vectors 
# u_test -= np.nanmean(u_test)
# v_test -= np.nanmean(v_test)

# reduce so it's not as crazy for the vectors 
reduction_factor = 3
u_vect = u_test[::reduction_factor,::reduction_factor]
v_vect = v_test[::reduction_factor,::reduction_factor]

u_vect = u_vect - np.nanmean(u_vect)
v_vect = v_vect - np.nanmean(v_vect)
uvmag = np.sqrt(u_vect**2 + v_vect**2)

x = np.linspace(0,u_test.shape[1],u_vect.shape[1])
y = np.linspace(0,u_test.shape[0],u_vect.shape[0])
# y = np.flipud(y)

plt.figure()
plt.imshow(ems_test,vmin=0,vmax=0.1,alpha=0.5)
plt.quiver(x,y,u_vect,v_vect,pivot='tail',scale=100,angles='xy')

In [ ]:
# plot histograms for given points 

# set consistent colours
gb_colour   = 'darkturquoise'
sb_colour   = 'tomato'
ogb_colour  = 'darkorange'
gc_colour   = 'seagreen'

# percentile line
pc = 99.99

logbins = np.logspace(-3,1,100)

fig,ax=plt.subplots(3,1,sharex=True,sharey=True)
fig.set_size_inches(5,8)

# axis 0
ax[0].set_xlim(1e-3,1)
ax[0].set_ylim(1e-5,1000)

sub_fig_labels = ['(a)','(b)','(c)']

for i,idx in enumerate([1,2,3]):

    tdata = copy.deepcopy(ems_cube[:,:,step_idxs[idx]])

    ems_gbs = tdata[gbs]#/np.nanmean(tdata[gbs])
    ems_sbs = tdata[sbs]#/np.nanmean(tdata[sbs])
    ems_ogbs = tdata[ogbs]#/np.nanmean(tdata[ogbs])
    ems_gcs = tdata[np.invert(gbs)]#/np.nanmean(tdata[np.invert(gbs)])
    

    ax[i].hist(ems_gbs,logbins,density=True,histtype='step',color=gb_colour)
    ax[i].hist(ems_sbs,logbins,density=True,histtype='step',color=sb_colour)
    ax[i].hist(ems_ogbs,logbins,density=True,histtype='step',color=ogb_colour)
    ax[i].hist(ems_gcs,logbins,density=True,histtype='step',color=gc_colour)

    # # add some vertical lines
    ax[i].axvline(np.nanpercentile(ems_gbs,pc),linestyle='--',color=gb_colour)
    ax[i].axvline(np.nanpercentile(ems_sbs,pc),linestyle='--',color=sb_colour)
    ax[i].axvline(np.nanpercentile(ems_ogbs,pc),linestyle='--',color=ogb_colour)
    ax[i].axvline(np.nanpercentile(ems_gcs,pc),linestyle='--',color=gc_colour)

    # annotate 
    glob_e11 = e11_glob[step_idxs[idx]]*100

    
    ax[i].annotate(sub_fig_labels[i] + ' $ϵ_{xx}$ = '+str(glob_e11.round(2))+'%',
                   xy=(ax[0].get_xlim()[0]*1.2,ax[0].get_ylim()[0]*2),
                   fontsize=14,
                   bbox=dict(boxstyle="square,pad=0.1",
                             fc='white',
                             ec='white'))

    ax[i].set_xscale('log')
    ax[i].set_yscale('log')


    ax[i].grid()
    ax[i].grid(which='minor',color='0.9')

    


ax[-1].set_xlabel('Effective strain / -')
ax[1].set_ylabel('Probability density / -')



ax[0].legend(['All GBs','Σ3GBs', 'Other GBs','Grain cores'])
plt.tight_layout()

# plt.savefig('figures_for_paper/Figure_4/histograms.svg')

In [ ]:
# plot histograms for given points - two x two grid for four figures

# set consistent colours
gb_colour   = 'darkturquoise'
sb_colour   = 'tomato'
ogb_colour  = 'darkorange'
gc_colour   = 'seagreen'

# percentile line
pc = 50

logbins = np.logspace(-3,1,100)

fig,ax=plt.subplots(2,2,sharex=True,sharey=True)
fig.set_size_inches(8,5)

# axis 0
ax.flatten()[0].set_xlim(1e-3,1)
ax.flatten()[0].set_ylim(1e-5,1000)

sub_fig_labels = ['(i)','(ii)','(iii)','(iv)']

for i,idx in enumerate([1,2,3,4]):

    tdata = copy.deepcopy(ems_cube[:,:,step_idxs[idx]])

    ems_gbs = tdata[gbs]#/np.nanmean(tdata[gbs])
    ems_sbs = tdata[sbs]#/np.nanmean(tdata[sbs])
    ems_ogbs = tdata[ogbs]#/np.nanmean(tdata[ogbs])
    ems_gcs = tdata[np.invert(gbs)]#/np.nanmean(tdata[np.invert(gbs)])
    

    ax.flatten()[i].hist(ems_gbs,logbins,density=True,histtype='step',color=gb_colour)
    ax.flatten()[i].hist(ems_sbs,logbins,density=True,histtype='step',color=sb_colour)
    ax.flatten()[i].hist(ems_ogbs,logbins,density=True,histtype='step',color=ogb_colour)
    ax.flatten()[i].hist(ems_gcs,logbins,density=True,histtype='step',color=gc_colour)

    # # add some vertical lines
    ax.flatten()[i].axvline(np.nanpercentile(ems_gbs,pc),linestyle='--',color=gb_colour)
    ax.flatten()[i].axvline(np.nanpercentile(ems_sbs,pc),linestyle='--',color=sb_colour)
    ax.flatten()[i].axvline(np.nanpercentile(ems_ogbs,pc),linestyle='--',color=ogb_colour)
    ax.flatten()[i].axvline(np.nanpercentile(ems_gcs,pc),linestyle='--',color=gc_colour)

    # annotate 
    glob_e11 = e11_glob[step_idxs[idx]]*100

    
    ax.flatten()[i].annotate(sub_fig_labels[i] + ' $ϵ_{xx}$ = '+str(glob_e11.round(2))+'%',
                   xy=(ax.flatten()[0].get_xlim()[0]*1.2,ax.flatten()[0].get_ylim()[0]*2.5),
                   fontsize=14,
                   bbox=dict(boxstyle="square,pad=0.1",
                             fc='white',
                             ec='white'))

    ax.flatten()[i].set_xscale('log')
    ax.flatten()[i].set_yscale('log')


    ax.flatten()[i].grid()
    ax.flatten()[i].grid(which='minor',color='0.9')

    


fig.supxlabel('Effective strain / -')
fig.supylabel('Probability density / -')



ax.flatten()[1].legend(['All GBs','Σ3GBs', 'Other GBs','Grain cores'],loc='upper right')
plt.tight_layout()

# plt.savefig('figures_for_paper/Figure_4/histograms.svg')

In [ ]:
# plot histograms for given points - two x two grid for four figures - linear scale

# set consistent colours
gb_colour   = 'darkturquoise'
sb_colour   = 'tomato'
ogb_colour  = 'darkorange'
gc_colour   = 'seagreen'

# percentile line
pc = 99

# logbins = np.logspace(-3,1,100)
linbins = np.linspace(0,1,20)

fig,ax=plt.subplots(2,2,sharex=True,sharey=True)
fig.set_size_inches(8,5)

# axis 0
ax.flatten()[0].set_xlim(0,0.4)
ax.flatten()[0].set_ylim(0,1000)

sub_fig_labels = ['(i)','(ii)','(iii)','(iv)']

for i,idx in enumerate([1,2,3,4]):

    tdata = copy.deepcopy(ems_cube[:,:,step_idxs[idx]])

    ems_gbs = tdata[gbs]#/np.nanmean(tdata[gbs])
    ems_sbs = tdata[sbs]#/np.nanmean(tdata[sbs])
    ems_ogbs = tdata[ogbs]#/np.nanmean(tdata[ogbs])
    ems_gcs = tdata[np.invert(gbs)]#/np.nanmean(tdata[np.invert(gbs)])
    

    ax.flatten()[i].hist(ems_gbs,linbins,density=True,histtype='step',color=gb_colour)
    ax.flatten()[i].hist(ems_sbs,linbins,density=True,histtype='step',color=sb_colour)
    ax.flatten()[i].hist(ems_ogbs,linbins,density=True,histtype='step',color=ogb_colour)
    ax.flatten()[i].hist(ems_gcs,linbins,density=True,histtype='step',color=gc_colour)

    # # add some vertical lines
    ax.flatten()[i].axvline(np.nanpercentile(ems_gbs,pc),linestyle='--',color=gb_colour)
    ax.flatten()[i].axvline(np.nanpercentile(ems_sbs,pc),linestyle='--',color=sb_colour)
    ax.flatten()[i].axvline(np.nanpercentile(ems_ogbs,pc),linestyle='--',color=ogb_colour)
    ax.flatten()[i].axvline(np.nanpercentile(ems_gcs,pc),linestyle='--',color=gc_colour)

    # annotate 
    glob_e11 = e11_glob[step_idxs[idx]]*100

    
    # ax.flatten()[i].annotate(sub_fig_labels[i] + ' $ϵ_{xx}$ = '+str(glob_e11.round(2))+'%',
    #                xy=(ax.flatten()[0].get_xlim()[0]*1.2,ax.flatten()[0].get_ylim()[0]*2.5),
    #                fontsize=14,
    #                bbox=dict(boxstyle="square,pad=0.1",
    #                          fc='white',
    #                          ec='white'))

    ax.flatten()[i].set_xscale('linear')
    ax.flatten()[i].set_yscale('linear')


    ax.flatten()[i].grid()
    ax.flatten()[i].grid(which='minor',color='0.9')

    


fig.supxlabel('Effective strain / -')
fig.supylabel('Probability density / -')



ax.flatten()[1].legend(['All GBs','Σ3GBs', 'Other GBs','Grain cores'],loc='upper right')
plt.tight_layout()

# plt.savefig('figures_for_paper/Figure_4/histograms.svg')

In [ ]:
# plot histograms for given points - two x two grid for four figures - rotation mag

# set consistent colours
gb_colour   = 'darkturquoise'
sb_colour   = 'tomato'
ogb_colour  = 'darkorange'
gc_colour   = 'seagreen'

# percentile line
pc = 99.99

logbins = np.logspace(-1,1.3,100)

fig,ax=plt.subplots(2,2,sharex=True,sharey=True)
fig.set_size_inches(8,5)

# axis 0
ax.flatten()[0].set_xlim(1e-1,10**1.5)
ax.flatten()[0].set_ylim(1e-5,1000)

sub_fig_labels = ['(i)','(ii)','(iii)','(iv)']

for i,idx in enumerate([1,2,3,4]):

    tdata = copy.deepcopy(abs(r_ang_cube[:,:,step_idxs[idx]]))

    r_ang_gbs = tdata[gbs]#/np.nanmean(tdata[gbs])
    r_ang_sbs = tdata[sbs]#/np.nanmean(tdata[sbs])
    r_ang_ogbs = tdata[ogbs]#/np.nanmean(tdata[ogbs])
    r_ang_gcs = tdata[np.invert(gbs)]#/np.nanmean(tdata[np.invert(gbs)])
    

    ax.flatten()[i].hist(r_ang_gbs,logbins,density=True,histtype='step',color=gb_colour)
    ax.flatten()[i].hist(r_ang_sbs,logbins,density=True,histtype='step',color=sb_colour)
    ax.flatten()[i].hist(r_ang_ogbs,logbins,density=True,histtype='step',color=ogb_colour)
    ax.flatten()[i].hist(r_ang_gcs,logbins,density=True,histtype='step',color=gc_colour)

    # # add some vertical lines
    ax.flatten()[i].axvline(np.nanpercentile(r_ang_gbs,pc),linestyle='--',color=gb_colour)
    ax.flatten()[i].axvline(np.nanpercentile(r_ang_sbs,pc),linestyle='--',color=sb_colour)
    ax.flatten()[i].axvline(np.nanpercentile(r_ang_ogbs,pc),linestyle='--',color=ogb_colour)
    ax.flatten()[i].axvline(np.nanpercentile(r_ang_gcs,pc),linestyle='--',color=gc_colour)

    # annotate 
    glob_e11 = e11_glob[step_idxs[idx]]*100

    
    ax.flatten()[i].annotate(sub_fig_labels[i] + ' $ϵ_{xx}$ = '+str(glob_e11.round(2))+'%',
                   xy=(ax.flatten()[0].get_xlim()[0]*1.2,ax.flatten()[0].get_ylim()[0]*2.5),
                   fontsize=14,
                   bbox=dict(boxstyle="square,pad=0.1",
                             fc='white',
                             ec='white'))

    ax.flatten()[i].set_xscale('log')
    ax.flatten()[i].set_yscale('log')


    ax.flatten()[i].grid()
    ax.flatten()[i].grid(which='minor',color='0.9')

    


fig.supxlabel('Effective strain / -')
fig.supylabel('Probability density / -')



ax.flatten()[1].legend(['All GBs','Σ3GBs', 'Other GBs','Grain cores'],loc='upper right')
plt.tight_layout()

# plt.savefig('figures_for_paper/Figure_4/histograms.svg')